In [2]:
import h5py
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from a5py import Ascot
from scipy.optimize import curve_fit
import matplotlib.cm as cm
from desc.equilibrium import EquilibriaFamily, Equilibrium
from desc.compute import get_params, get_profiles, get_transforms
from desc.vmec_utils import ptolemy_linear_transform
from desc.grid import LinearGrid, QuadratureGrid
from desc.plotting import *
import desc.plotting as plotting

import desc
import desc.io as descio
from desc.grid import LinearGrid, Grid
import numpy as np

ERROR:2026-03-12 06:56:47,183:jax._src.xla_bridge:487: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/global/homes/m/mpatel26/.conda/envs/ascot-develop-mp/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 485, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/global/homes/m/mpatel26/.conda/envs/ascot-develop-mp/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/global/homes/m/mpatel26/.conda/envs/ascot-develop-mp/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE


In [19]:
input_dir = "/pscratch/sd/m/mpatel26/ascot_h5s/"
ascot = Ascot(input_dir + "G1600_fixed_alpha-analysis_simmode3.h5")

In [20]:
info = ascot.data.ls()

Inputs: [only active shown]
options opt        1830569440 2026-03-12 06:38:44
TAG
+ 1 other(s)
bfield  B_STS      1494509224 2026-03-11 10:53:12
TAG
+ 0 other(s)
efield  E_TC       2732785348 2026-03-11 10:53:20
TAG
+ 0 other(s)
marker  gc         1156243561 2026-03-11 10:53:26
TAG
+ 0 other(s)
plasma  plasma_1D  0549144311 2026-03-11 10:53:16
TAG
+ 0 other(s)
neutral N0_1D      2000803159 2026-03-11 10:53:20
TAG
+ 0 other(s)
wall    wall_3D    3525230836 2026-03-11 10:53:20
TAG
+ 0 other(s)
boozer  Boozer     4283470427 2026-03-11 10:53:20
TAG
+ 0 other(s)
mhd     MHD_STAT   3980722808 2026-03-11 10:53:20
TAG
+ 0 other(s)
asigma  asigma_loc 0082373770 2026-03-11 10:53:20
TAG
+ 0 other(s)
Results:
run        0811467644 2026-03-12 08:30:59. [active]
Testing
run        1337675255 2026-03-11 14:47:26.
Testing



In [ ]:
option = ascot.data.options.active.read()
print(option.keys())
print(option['SIM_MODE'])
print(option['ENDCOND_MAX_RHO'])
print(option['ENABLE_FLR_LOSSES'])


dict_keys(['ADAPTIVE_MAX_DPHI', 'ADAPTIVE_MAX_DRHO', 'ADAPTIVE_TOL_CCOL', 'ADAPTIVE_TOL_ORBIT', 'DISABLE_ENERGY_CCOLL', 'DISABLE_FIRSTORDER_GCTRANS', 'DISABLE_GCDIFF_CCOLL', 'DISABLE_PITCH_CCOLL', 'DIST_MAX_CHARGE', 'DIST_MAX_EKIN', 'DIST_MAX_MU', 'DIST_MAX_PHI', 'DIST_MAX_PPA', 'DIST_MAX_PPE', 'DIST_MAX_PPHI', 'DIST_MAX_PR', 'DIST_MAX_PTOR', 'DIST_MAX_PZ', 'DIST_MAX_R', 'DIST_MAX_RHO', 'DIST_MAX_THETA', 'DIST_MAX_TIME', 'DIST_MAX_Z', 'DIST_MIN_CHARGE', 'DIST_MIN_EKIN', 'DIST_MIN_MU', 'DIST_MIN_PHI', 'DIST_MIN_PPA', 'DIST_MIN_PPE', 'DIST_MIN_PPHI', 'DIST_MIN_PR', 'DIST_MIN_PTOR', 'DIST_MIN_PZ', 'DIST_MIN_R', 'DIST_MIN_RHO', 'DIST_MIN_THETA', 'DIST_MIN_TIME', 'DIST_MIN_Z', 'DIST_NBIN_CHARGE', 'DIST_NBIN_EKIN', 'DIST_NBIN_MU', 'DIST_NBIN_PHI', 'DIST_NBIN_PPA', 'DIST_NBIN_PPE', 'DIST_NBIN_PPHI', 'DIST_NBIN_PR', 'DIST_NBIN_PTOR', 'DIST_NBIN_PZ', 'DIST_NBIN_R', 'DIST_NBIN_RHO', 'DIST_NBIN_THETA', 'DIST_NBIN_TIME', 'DIST_NBIN_Z', 'ENABLE_ADAPTIVE', 'ENABLE_ALDFORCE', 'ENABLE_ATOMIC', 'ENABLE

In [16]:
option['SIM_MODE'] = 3
option['ENDCOND_MAX_RHO'] = 0.8  
option['ENABLE_FLR_LOSSES'] = 0
ascot.data.create_input('opt', **option, activate = True)
option = ascot.data.options.active.read()
print(option.keys())
print(option['SIM_MODE'])
print(option['ENDCOND_MAX_RHO'])
print(option['ENABLE_FLR_LOSSES'])


dict_keys(['ADAPTIVE_MAX_DPHI', 'ADAPTIVE_MAX_DRHO', 'ADAPTIVE_TOL_CCOL', 'ADAPTIVE_TOL_ORBIT', 'DISABLE_ENERGY_CCOLL', 'DISABLE_FIRSTORDER_GCTRANS', 'DISABLE_GCDIFF_CCOLL', 'DISABLE_PITCH_CCOLL', 'DIST_MAX_CHARGE', 'DIST_MAX_EKIN', 'DIST_MAX_MU', 'DIST_MAX_PHI', 'DIST_MAX_PPA', 'DIST_MAX_PPE', 'DIST_MAX_PPHI', 'DIST_MAX_PR', 'DIST_MAX_PTOR', 'DIST_MAX_PZ', 'DIST_MAX_R', 'DIST_MAX_RHO', 'DIST_MAX_THETA', 'DIST_MAX_TIME', 'DIST_MAX_Z', 'DIST_MIN_CHARGE', 'DIST_MIN_EKIN', 'DIST_MIN_MU', 'DIST_MIN_PHI', 'DIST_MIN_PPA', 'DIST_MIN_PPE', 'DIST_MIN_PPHI', 'DIST_MIN_PR', 'DIST_MIN_PTOR', 'DIST_MIN_PZ', 'DIST_MIN_R', 'DIST_MIN_RHO', 'DIST_MIN_THETA', 'DIST_MIN_TIME', 'DIST_MIN_Z', 'DIST_NBIN_CHARGE', 'DIST_NBIN_EKIN', 'DIST_NBIN_MU', 'DIST_NBIN_PHI', 'DIST_NBIN_PPA', 'DIST_NBIN_PPE', 'DIST_NBIN_PPHI', 'DIST_NBIN_PR', 'DIST_NBIN_PTOR', 'DIST_NBIN_PZ', 'DIST_NBIN_R', 'DIST_NBIN_RHO', 'DIST_NBIN_THETA', 'DIST_NBIN_TIME', 'DIST_NBIN_Z', 'ENABLE_ADAPTIVE', 'ENABLE_ALDFORCE', 'ENABLE_ATOMIC', 'ENABLE

In [18]:
info = ascot.data.options.ls()

opt        0871731363 2026-03-12 07:14:16 [active]
TAG
opt        1179249111 2026-03-12 07:10:07
TAG

